# Datacenter Infrastructure Graph

Load a datacenter graph with **Datacenters**, **Racks**, **Servers**, and **Applications**.

Run each cell top to bottom.

In [ ]:
# GCS setup — skipped automatically in production
try:
    import pandas as pd, numpy as np, requests, os, random
    
    CP = 'http://control-plane:8080'
    GCS = 'http://fake-gcs-local:4443'
    BUCKET = 'graph-olap-local-dev'
    OWNER = 'demo@example.com'
    HEADERS = {'Content-Type': 'application/json', 'X-Username': OWNER, 'X-User-Role': 'admin'}
    
    requests.post(f'{GCS}/storage/v1/b', json={'name': BUCKET})
    print('Connected!')except Exception as _e:
    print(f'GCS setup skipped (production mode): {_e}')


## 1. Generate Data

In [ ]:
random.seed(42)

dcs = pd.DataFrame({
    'dc_id': [f'dc-{i:02d}' for i in range(1, 6)],
    'name': ['London-1', 'Frankfurt-1', 'Virginia-1', 'Mumbai-1', 'Singapore-1'],
    'location': ['London, UK', 'Frankfurt, DE', 'Ashburn, US', 'Mumbai, IN', 'Singapore, SG'],
    'tier': ['Tier 4', 'Tier 3', 'Tier 4', 'Tier 3', 'Tier 3'],
})

rack_rows = []
rack_num = 1
for dc in dcs['dc_id']:
    for row in ['A', 'B', 'C']:
        for pos in range(1, 5):
            rack_rows.append({'rack_id': f'rack-{rack_num:03d}', 'dc_id': dc, 'row': row, 'position': pos})
            rack_num += 1
racks = pd.DataFrame(rack_rows)

srv_rows = []
srv_num = 1
for rack_id in racks['rack_id']:
    for _ in range(random.randint(3, 5)):
        srv_rows.append({'server_id': f'srv-{srv_num:04d}', 'rack_id': rack_id,
            'hostname': f'node-{srv_num:04d}.internal',
            'cpu_cores': random.choice([16, 32, 64, 128]),
            'ram_gb': random.choice([64, 128, 256, 512]),
            'status': random.choice(['running']*4 + ['maintenance', 'offline'])})
        srv_num += 1
servers = pd.DataFrame(srv_rows)

apps = pd.DataFrame({
    'app_id': [f'app-{i:02d}' for i in range(1, 11)],
    'name': ['Auth Service', 'Payment Gateway', 'User API', 'Analytics Pipeline', 'ML Training',
             'Dashboard UI', 'Notification Service', 'Search Index', 'CI/CD Runner', 'Monitoring Stack'],
    'team': ['Platform', 'Payments', 'Platform', 'Data', 'AI', 'Frontend', 'Platform', 'Data', 'DevOps', 'SRE'],
    'criticality': ['Critical', 'Critical', 'High', 'Medium', 'Low', 'High', 'Medium', 'High', 'Medium', 'Critical'],
})

located_in = racks[['rack_id', 'dc_id']].copy()
installed_in = servers[['server_id', 'rack_id']].copy()
hosted_rows = []
for app_id in apps['app_id']:
    for srv in random.sample(list(servers['server_id']), random.randint(2, 5)):
        hosted_rows.append({'app_id': app_id, 'server_id': srv})
hosted_on = pd.DataFrame(hosted_rows)

print(f'{len(dcs)} datacenters, {len(racks)} racks, {len(servers)} servers, {len(apps)} apps')
print(f'{len(located_in)} located_in, {len(installed_in)} installed_in, {len(hosted_on)} hosted_on')
dcs

## 2. Create Mapping & Instance

In [ ]:
# Create mapping
mapping = requests.post(f'{CP}/api/mappings', headers=HEADERS, json={
    'name': 'Datacenter Infrastructure',
    'description': '[via:jupyter] DC infra graph',
    'node_definitions': [
        {'label': 'Datacenter', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'dc_id', 'type': 'STRING'}, 'properties': [{'name': 'name', 'type': 'STRING'}, {'name': 'location', 'type': 'STRING'}, {'name': 'tier', 'type': 'STRING'}]},
        {'label': 'Rack', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'rack_id', 'type': 'STRING'}, 'properties': [{'name': 'dc_id', 'type': 'STRING'}, {'name': 'row', 'type': 'STRING'}, {'name': 'position', 'type': 'INT64'}]},
        {'label': 'Server', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'server_id', 'type': 'STRING'}, 'properties': [{'name': 'rack_id', 'type': 'STRING'}, {'name': 'hostname', 'type': 'STRING'}, {'name': 'cpu_cores', 'type': 'INT64'}, {'name': 'ram_gb', 'type': 'INT64'}, {'name': 'status', 'type': 'STRING'}]},
        {'label': 'Application', 'sql': 'SELECT * FROM placeholder', 'primary_key': {'name': 'app_id', 'type': 'STRING'}, 'properties': [{'name': 'name', 'type': 'STRING'}, {'name': 'team', 'type': 'STRING'}, {'name': 'criticality', 'type': 'STRING'}]},
    ],
    'edge_definitions': [
        {'type': 'LOCATED_IN', 'from_node': 'Rack', 'to_node': 'Datacenter', 'sql': 'SELECT * FROM placeholder', 'from_key': 'rack_id', 'to_key': 'dc_id', 'properties': []},
        {'type': 'INSTALLED_IN', 'from_node': 'Server', 'to_node': 'Rack', 'sql': 'SELECT * FROM placeholder', 'from_key': 'server_id', 'to_key': 'rack_id', 'properties': []},
        {'type': 'HOSTED_ON', 'from_node': 'Application', 'to_node': 'Server', 'sql': 'SELECT * FROM placeholder', 'from_key': 'app_id', 'to_key': 'server_id', 'properties': []},
    ]
}).json()
mapping_id = mapping['data']['id']
print(f'Mapping: {mapping_id}')

# Create instance
inst = requests.post(f'{CP}/api/instances', headers=HEADERS, json={
    'mapping_id': mapping_id, 'name': 'Datacenter Infra',
    'description': '[via:jupyter]', 'wrapper_type': 'falkordb', 'ttl': 'PT4H',
}).json()
instance_id = inst['data']['id']
snapshot_id = inst['data']['snapshot_id']
print(f'Instance: {instance_id}, Snapshot: {snapshot_id}')

In [ ]:
# Save parquet and upload to GCS
import time
base = '/tmp/dc'
for d in ['nodes/Datacenter','nodes/Rack','nodes/Server','nodes/Application',
          'edges/LOCATED_IN','edges/INSTALLED_IN','edges/HOSTED_ON']:
    os.makedirs(f'{base}/{d}', exist_ok=True)

dcs.to_parquet(f'{base}/nodes/Datacenter/data.parquet', index=False)
racks.to_parquet(f'{base}/nodes/Rack/data.parquet', index=False)
servers.to_parquet(f'{base}/nodes/Server/data.parquet', index=False)
apps.to_parquet(f'{base}/nodes/Application/data.parquet', index=False)
located_in.to_parquet(f'{base}/edges/LOCATED_IN/data.parquet', index=False)
installed_in.to_parquet(f'{base}/edges/INSTALLED_IN/data.parquet', index=False)
hosted_on.to_parquet(f'{base}/edges/HOSTED_ON/data.parquet', index=False)

prefix = f'{OWNER}/{mapping_id}/v1/{snapshot_id}'
for name in ['nodes/Datacenter','nodes/Rack','nodes/Server','nodes/Application',
             'edges/LOCATED_IN','edges/INSTALLED_IN','edges/HOSTED_ON']:
    with open(f'{base}/{name}/data.parquet', 'rb') as f:
        requests.post(f'{GCS}/upload/storage/v1/b/{BUCKET}/o?uploadType=media&name={prefix}/{name}/data.parquet',
                      data=f.read(), headers={'Content-Type': 'application/octet-stream'})
print('Data uploaded!')

# Wait for instance to start
for i in range(60):
    status = requests.get(f'{CP}/api/instances/{instance_id}', headers=HEADERS).json()['data']['status']
    if status == 'running': break
    time.sleep(5)

pod_name = requests.get(f'{CP}/api/instances/{instance_id}', headers=HEADERS).json()['data']['pod_name']
W = f'http://{pod_name}:8000'
print(f'Instance running at {W}')

## 3. Query

In [ ]:
def q(cypher):
    r = requests.post(f'{W}/query', json={'query': cypher})
    d = r.json()
    return pd.DataFrame(d['rows'], columns=d['columns'])

q('MATCH (n) RETURN labels(n)[0] AS type, count(n) AS count ORDER BY count DESC')

In [ ]:
# Servers per datacenter
q('''
MATCH (s:Server)-[:INSTALLED_IN]->(r:Rack)-[:LOCATED_IN]->(dc:Datacenter)
RETURN dc.name AS datacenter, dc.location AS location, count(s) AS servers
ORDER BY servers DESC
''')

In [ ]:
# Critical apps
q('''
MATCH (a:Application)-[:HOSTED_ON]->(s:Server)
WHERE a.criticality = 'Critical'
RETURN a.name AS app, a.team AS team, count(s) AS servers
ORDER BY servers DESC
''')

In [ ]:
# Offline servers
q('''
MATCH (s:Server)-[:INSTALLED_IN]->(r:Rack)-[:LOCATED_IN]->(dc:Datacenter)
WHERE s.status = 'offline'
RETURN s.hostname AS server, dc.name AS datacenter
''')

In [ ]:
# Impact: if rack-001 goes down, which apps are affected?
q('''
MATCH (a:Application)-[:HOSTED_ON]->(s:Server)-[:INSTALLED_IN]->(r:Rack)
WHERE r.rack_id = 'rack-001'
RETURN DISTINCT a.name AS affected_app, a.criticality AS criticality, count(s) AS servers_on_rack
ORDER BY criticality
''')

In [ ]:
# Total compute per datacenter
q('''
MATCH (s:Server)-[:INSTALLED_IN]->(r:Rack)-[:LOCATED_IN]->(dc:Datacenter)
RETURN dc.name AS datacenter, sum(s.cpu_cores) AS total_cores, sum(s.ram_gb) AS total_ram_gb
ORDER BY total_cores DESC
''')

## 4. Visualize Graph

In [ ]:
from graph_olap_sdk import visualize

visualize(W, limit=200, title='Datacenter Infrastructure')

## Cleanup

In [ ]:
# Uncomment to terminate:
# requests.delete(f'{CP}/api/instances/{instance_id}', headers=HEADERS)
# print('Done!')